In [ ]:
import numpy as np
import pandas as pd

In [ ]:
import os
import json
import gzip
import pandas as pd
from urllib.request import urlopen

In [ ]:
def parse(path):
  g = gzip.open(path, 'r')
  for l in g:
    yield json.loads(l)

In [ ]:
import pandas as pd
import gzip
import json

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
  return pd.DataFrame.from_dict(df, orient='index')

df = getDF('/data/zyang/datasets/amazon_book/Books_5.json.gz')

In [ ]:
df.head(2)

In [ ]:
metadata = []
with gzip.open('/data/zyang/datasets/amazon_book/meta_Books.json.gz') as f:
    for l in f:
        metadata.append(json.loads(l.strip()))
    
# total length of list, this number equals total number of products
print(len(metadata))

# first row of the list
print(metadata[0])

# convert list into pandas dataframe

df_meta = pd.DataFrame.from_dict(metadata)

print(len(df_meta))

In [ ]:
df3 = df_meta.fillna('')
df4 = df3[df3.title.str.contains('getTime')] # unformatted rows
df5 = df3[~df3.title.str.contains('getTime')] # filter those unformatted rows
print(len(df4))
print(len(df5))

In [ ]:
df['overall'].dropna().describe()

In [ ]:
df.columns

In [ ]:
rating = df[['overall','reviewTime','reviewerID','asin','vote','unixReviewTime']]
rating.head(2) 

In [ ]:
rating = rating[['asin','reviewerID','overall','unixReviewTime']]
rating.columns = ['asin','user','rating','timestamp']

In [ ]:
df5.columns

In [ ]:
meta_data = df5[['asin','category','title','brand','price']]
meta_data.head(2)

In [ ]:
meta_data.shape

In [ ]:
meta_data = meta_data.drop_duplicates(subset=['asin'],keep='last')
meta_data.shape

In [ ]:
meta_data.head(2)

In [ ]:
data = rating.merge(meta_data,on='asin',how='right')

In [ ]:
data.head(2)

In [ ]:
data.shape

In [ ]:
data = data.dropna()

In [ ]:
data.shape

In [ ]:
data.head(5)

In [ ]:
rating_ = data.copy()
rating_.head(2)

In [ ]:
rating_.columns = ['iid','uid','rating','timestamp','category','title','brand','price']
rating_.head(2)

In [ ]:
# rating_ = pd.merge(rating,item_info,on='iid',how='inner')

In [ ]:
date_min = pd.to_datetime(rating_.timestamp,unit='s').min() 
date_max = pd.to_datetime(rating_.timestamp,unit='s').max()
date_min,date_max

In [ ]:
rating_.shape

In [ ]:
date_gap = (date_max-date_min)//(19*2)
date_gap

In [ ]:
rating_['time'] = pd.to_datetime(rating_.timestamp, unit='s').map(lambda x: x.year )

In [ ]:
rating_.head(2)

In [ ]:
# rating_['time'] = rating_['time'] - rating_['time'].min()

In [ ]:
import numpy as np
np.sort(rating_.time.unique())

In [ ]:
rating_.uid.unique().shape, rating_.iid.unique().shape,

In [ ]:
s_rating = rating_[rating_.time.isin([2017])].copy()
s_rating['time'] = pd.to_datetime(s_rating.timestamp, unit='s').map(lambda x: x.month)
s_rating = s_rating[s_rating.time.isin(range(1,13))]


In [ ]:
s_rating.shape

In [ ]:
s_rating.groupby('uid').agg({"rating":'count'}).describe()

In [ ]:
item_info = s_rating.groupby('iid').agg({"rating":'count'})
user_info = s_rating.groupby('uid').agg({"rating":'count'})

In [ ]:
active_item = item_info[item_info['rating']>20].index #.sample(frac=10/20,random_state=2023).index
active_user = user_info[user_info['rating']>20].index #.sample(frac=10/20,random_state=2023).index
active_item.shape, active_user.shape

In [ ]:
s_rating = s_rating[s_rating['uid'].isin(active_user)]
s_rating = s_rating[s_rating['iid'].isin(active_item)]
s_rating.shape

In [ ]:
item_info = s_rating.groupby('iid').agg({"rating":'count'})
user_info = s_rating.groupby('uid').agg({"rating":'count'})
user_info.mean(), item_info.mean()

In [ ]:
s_rating.shape, s_rating.uid.unique().shape, s_rating.iid.unique().shape

In [ ]:
s_rating.uid.unique().shape, s_rating.iid.unique().shape

In [ ]:
s_rating = s_rating.reset_index()
s_rating.head(2)

In [ ]:
s_rating.groupby('time').agg({'rating':'count'}).reset_index().plot(x='time',kind='bar')

In [ ]:
s_rating.rating.unique()

In [ ]:
s_rating['label'] = s_rating['rating'].apply(lambda x: 1 if x>=5 else 0)
s_rating['label'].describe()

In [ ]:
users = s_rating.uid.unique()
items = s_rating.iid.unique()
users_map = dict(zip(users, np.arange(users.shape[0])+1))
items_map = dict(zip(items, np.arange(items.shape[0])+1))
# users_map[0]=0
# items_map[0]=0
s_rating['uid'] = s_rating['uid'].map(users_map)
s_rating['iid'] = s_rating['iid'].map(items_map)
s_rating.uid.max(), s_rating.iid.max()

In [ ]:
rating_train = s_rating[s_rating.time.isin(range(1,12))].copy()
rating_valid_test = s_rating[s_rating.time.isin([12])].copy()
rating_valid_test.sort_values(by="timestamp",inplace=True)
N_ = rating_valid_test.shape[0]//2
# rating_p = rating_valid_test.iloc[0:N_].copy()
# rating_train = pd.concat([rating_train, rating_p],axis=0)
rating_valid = rating_valid_test.iloc[:N_].copy()
rating_test = rating_valid_test.iloc[N_:].copy()
rating_train.shape, rating_valid.shape, rating_test.shape

In [ ]:
rating_valid_test.timestamp.values[0:5].argsort()

In [ ]:
rating_train.time.unique(),rating_valid.time.unique()

In [ ]:
s_rating[s_rating.time.isin(range(7,13))].groupby('iid').agg({"rating":'count'}).mean()

In [ ]:
train_user = rating_train['uid'].unique()
train_item = rating_train['iid'].unique()
rating_valid['not_cold'] = rating_valid[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")
rating_test['not_cold'] = rating_test[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")

In [ ]:
rating_valid['not_cold'].describe(), rating_test['not_cold'].describe()

In [ ]:
rating_train.shape, rating_valid.shape, rating_test.shape

In [ ]:
rating_train.groupby("uid").agg({'label':['count','sum']}).describe(), rating_train.groupby("iid").agg({'label':['count','sum']}).describe()

In [ ]:
# rating_valid.groupby("uid").agg({'label':['count','sum']}).describe()
rating_test.groupby("uid").agg({'label':['count','sum']}).describe(), rating_test.groupby("iid").agg({'label':['count','sum']}).describe()
# rating_valid_test.groupby("uid").agg({'label':['count','sum']}).describe()

In [ ]:
# train_slot = list(range(26,36))
# valid_slot = list(range(36,37))
# test_slot = list(range(37,38))
# rating_  = rating_all[rating_all['time'].isin(list(range(26,38)))].copy().reset_index()
# train_slot = list(range(2010,2017))
# valid_slot = list(range(34,36))
# test_slot = list(range(36,38))
# rating_  = rating_all[rating_all['time'].isin(list(range(24,38)))].copy().reset_index()

In [ ]:
# rating_['rating'].unique()

In [ ]:
# rating_['label'] = rating_['rating'].apply(lambda x: 1 if x>4 else 0)
# rating_.head(5)

In [ ]:
# rating_.label.describe()

In [ ]:
# rating_.uid.min(), rating_.uid.max(), rating_.iid.min(),rating_.iid.max()

In [ ]:
# rating_.describe()

In [ ]:
# rating_.shape[0]/(rating_.uid.max()*rating_.iid.max())

## split by time order

In [ ]:
# rating_train = rating_[rating_['time'].isin(train_slot)].copy()
# rating_valid = rating_[rating_['time'].isin(valid_slot)].copy()
# rating_test = rating_[rating_['time'].isin(test_slot)].copy()

In [ ]:
rating_train.shape, rating_valid.shape, rating_test.shape

In [ ]:
rating_valid_f = rating_valid
rating_test_f = rating_test

In [ ]:
def filter_cold_start(train,valid,test):
    train_user = train.uid.unique()
    train_item = train.iid.unique()
    valid = valid[valid['uid'].isin(train_user)]
    test = test[test['uid'].isin(train_user)]
    valid = valid[valid['iid'].isin(train_item)]
    test = test[test['iid'].isin(train_item)]
    return valid, test

In [ ]:
# rating_valid_f, rating_test_f = filter_cold_start(rating_train,rating_valid,rating_test)
# rating_valid_f.shape, rating_test_f.shape

In [ ]:
rating_train.label.mean(), rating_valid.label.mean(), rating_test.label.mean()

In [ ]:
rating_valid_f.label.mean(), rating_test_f.label.mean() 

In [ ]:
rating_valid_f.columns

In [ ]:
import copy
def deal_with_each_u(x,u):
    items = np.array(x.iid)
    labels = np.array(x.label)
    titles = np.array(x.title)
    timestamp = np.array(x.timestamp)
    flags =  np.array(x.flag) 
    his = [0] # adding a '0' by default
    his_title = ['']
    results = []
    for i in range(items.shape[0]):
        results.append((u, items[i], timestamp[i], np.array(his), copy.copy(his_title),titles[i], labels[i], flags[i]))
        # training data
        if labels[i] > 0: # positive 
            his.append(items[i])
            his_title.append(titles[i])
    return results

In [ ]:
rating_train = rating_train.copy()

In [ ]:
rating_train['flag'] =  pd.DataFrame(np.ones(rating_train.shape[0])*-1, index=rating_train.index)
rating_valid_f['flag'] = pd.DataFrame(np.zeros(rating_valid_f.shape[0]), index=rating_valid_f.index)
rating_test_f['flag'] = pd.DataFrame(np.ones(rating_test_f.shape[0]), index=rating_test_f.index)
data_ = pd.concat([rating_train, rating_valid_f, rating_test_f],axis=0,ignore_index=True)
data_ = data_.sort_values(by=['uid','timestamp'])
u_inter_all = data_.groupby('uid').agg({'iid':list, 'label':list, 'title':list, 'timestamp':list,'flag':list})


In [ ]:
data_.flag.unique()

In [ ]:
results = []
for u in u_inter_all.index:
    results.extend(deal_with_each_u(u_inter_all.loc[u],u))

In [ ]:
u_, i_, time_, label_, his_, his_title, title_,flag_ = [],[],[],[],[],[],[],[]
for re_ in results:
    u_.append(re_[0])
    i_.append(re_[1])
    time_.append(re_[2])
    his_.append(re_[3])
    his_title.append(re_[4])
    title_.append(re_[5])
    label_.append(re_[6])
    flag_.append(re_[7])

In [ ]:
data_ = pd.DataFrame({"uid":u_,'iid':i_,'label':label_, 'timestamp': time_ , 'his':his_,'his_title':his_title,'title':title_, 'flag': flag_})


In [ ]:
data_.head(2)

In [ ]:
data_.label.describe()

In [ ]:
data_.tail(5)

In [ ]:
train_ = data_[data_['flag'].isin([-1])].copy()
valid_ = data_[data_['flag'].isin([0])].copy()
test_ = data_[data_['flag'].isin([1])].copy()
train_.shape,valid_.shape,test_.shape

In [ ]:
train_user = train_['uid'].unique()
train_item = train_['iid'].unique()
valid_['not_cold'] = valid_[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")
test_['not_cold'] = test_[['uid','iid']].apply(lambda x: x.uid in train_user and x.iid in train_item, axis=1).astype("int")

In [ ]:
valid_['not_cold'].describe(), test_['not_cold'].describe()

In [ ]:
train_['not_cold'] = pd.DataFrame(np.ones(train_.shape[0]),index=train_.index).astype("int")
train_.head(2)

In [ ]:
save_path = "/data/zyang/datasets/amazon_book_new/"
train_.to_pickle(save_path+"train_ood2.pkl")
valid_.to_pickle(save_path+"valid_ood2.pkl")
test_.to_pickle(save_path+"test_ood2.pkl")

In [ ]:
train_.head(2)

In [ ]:
train_.title.iloc[0]

In [ ]:
valid_.shape

In [ ]:
valid_small = valid_.sample(frac=0.25,random_state=2023)
valid_small.to_pickle(save_path+"valid_small_ood2.pkl")

In [ ]:
valid_small.shape

In [ ]:
# valid_small.shape

In [ ]:
train_['his'].apply(len).describe()

In [ ]:
train_.label.mean(), test_.label.mean(), valid_.label.mean()  

In [ ]:
test_.groupby("uid").agg({"label":['count','sum']}).describe()

In [ ]:
train_.groupby("iid").agg({"label":['count','sum']}).describe()

In [ ]:
valid_[valid_['uid'].isin(train_['uid'].unique())].uid.unique().shape

In [ ]:
train_['iid'].unique()

In [ ]:
valid_[valid_['iid'].isin(train_['iid'].unique())].iid.unique().shape

In [ ]:
valid_.iid.unique().shape, test_.iid.unique().shape

In [ ]:
valid_.uid.unique().shape, test_.uid.unique().shape

In [ ]:
train_.uid.unique().shape, train_.iid.unique().shape

In [ ]:

np.setdiff1d(test_.uid.unique(),train_.uid.unique()).shape, np.setdiff1d(valid_.uid.unique(),train_.uid.unique()).shape

In [ ]:
np.setdiff1d(test_.iid.unique(),train_.iid.unique()).shape, np.setdiff1d(valid_.iid.unique(),train_.iid.unique()).shape

In [ ]:
# m = train_.groupby('uid').agg({'label':'count'}).sort_values('label').reset_index()
# m.plot(x='uid',kind='bar')

In [ ]:
# train_.uid.max()



In [ ]:
# save_path = "/data/zyang/datasets/ml-1m/"
# train_ = pd.read_pickle(save_path+"train_ood2.pkl")
# valid_ = pd.read_pickle(save_path+"valid_ood2.pkl")
# test_ = pd.read_pickle(save_path+"test_ood2.pkl")

In [ ]:
max(train_.uid.max(),test_.uid.max(),valid_.uid.max())+1, max(train_.iid.max(),test_.iid.max(),valid_.iid.max())+1


In [ ]:
# valid_small = pd.read_pickle(save_path+"valid_small_ood2.pkl")

In [ ]:
# valid_small.uid.max(), valid_small.iid.max()

In [ ]:
import pandas as pd

In [ ]:
# save_path = "/data/zyang/datasets/amazon/"
# train = pd.read_pickle(save_path+"train_ood2.pkl")
# valid = pd.read_pickle(save_path+"valid_ood2.pkl")
# test = pd.read_pickle(save_path+"test_ood2.pkl")

In [ ]:
valid_.groupby("uid").agg({"label":['count','mean']}).describe()

In [ ]:
test_.groupby("uid").agg({"label":['count','mean']}).describe()

## read info

In [ ]:
import pandas as pd
save_path = "/data/zyang/datasets/amazon_book_new/"
train_ = pd.read_pickle(save_path+"train_ood2.pkl")
valid_ = pd.read_pickle(save_path+"valid_ood2.pkl")
test_ = pd.read_pickle(save_path+"test_ood2.pkl")
data = pd.concat([train_,valid_,test_],axis=0)

In [ ]:
data.groupby("uid").agg({"label":['sum','count']}).describe()